In [1]:
from gettext import npgettext
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.colors as pc
from IPython.display import HTML
import numpy as np
import polars as pl
HTML("""
<link href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:wght@400;700&display=swap" rel="stylesheet">
""")

In [2]:
DATA = "./qc1.csv"
SPECIES = "H_cyanocinctus"
METRICS = ["busco_completeness", "n50", "aun", "kmer_QV", "kmer_completeness"]


def read_qc(path: Path, species: str) -> pd.DataFrame:
    df = pd.read_csv(path, na_values=["NA", ""])
    df["value"] = pd.to_numeric(df["value"].astype(str).str.replace(",", "", regex=False), errors="coerce")
    df["stage_order"] = pd.to_numeric(df["stage_order"], errors="coerce")
    return df.loc[
        (df["species"] == species)
        & (~df["stage"].isin(["original", "final"]))
        & (df["metric"].isin(METRICS)),
        ["stage", "stage_order", "metric", "value"],
    ]


df = read_qc(DATA, SPECIES)

wide = (
    df.pivot_table(
        index=["stage", "stage_order"],
        columns="metric",
        values="value",
        aggfunc="first",
    )
    .reset_index()
    .sort_values("stage_order")
    .reset_index(drop=True)
)

wide

metric,stage,stage_order,aun,busco_completeness,kmer_QV,kmer_completeness,n50
0,denovo,1,20292006.0,99.44,31.0,90.0,16171423.0
1,denovo_correct,2,20292168.7,99.44,32.0,90.0,16171549.0
2,denovo_break,3,20100922.8,99.43,32.0,90.0,16171549.0
3,denovo_polish,4,20078579.8,99.42,35.0,91.0,16160212.0
4,scaff,5,151914109.6,99.49,40.0,94.0,147776827.0
5,scaff_correct,6,149562311.3,99.49,45.0,95.0,138976891.0
6,man_curation,7,288080157.9,99.43,50.0,96.0,262685935.0
7,gap_fill,8,250000000.0,99.51,51.0,97.0,275000000.0
8,polish_final,9,265000000.0,99.77,53.0,98.0,264245889.0


In [3]:
AXIS_RANGES = {
    "busco_completeness": (90, 100),
}

ranges = wide[METRICS].agg(["min", "max"])

plot_ranges = {}
for metric in METRICS:
    if metric in AXIS_RANGES:
        plot_ranges[metric] = AXIS_RANGES[metric]
    else:
        plot_ranges[metric] = (
            ranges.loc["min", metric],
            ranges.loc["max", metric],
        )

scaled = wide.copy()

for metric in METRICS:
    lo, hi = plot_ranges[metric]
    scaled[metric] = 0.5 if hi == lo else (wide[metric] - lo) / (hi - lo)
    scaled[metric] = scaled[metric].clip(0, 1)

n = len(METRICS)
angles = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, n, endpoint=False)

def xy(values):
    values = np.asarray(values)
    return values * np.cos(angles), values * np.sin(angles)

STAGE_COLORMAP = "Bluered"
FILL_OPACITY   = 0.1
LABEL_R        = 1.4
LEADER_INNER   = 1.10
LEADER_OUTER   = 1.32

# Define METRIC_LABELS dictionary
METRIC_LABELS = {
    "busco_completeness": "BUSCO Completeness",
    "n50": "N50 (Mb)",
    "aun": "AUN (Mb)",
    "kmer_QV": "Kmer QV",
    "kmer_completeness": "Kmer Completeness",
}

fig = go.Figure()

# Grid rings
for r in [0.25, 0.5, 0.75, 1.0, 1.1]:
    x, y = xy([r] * n)
    fig.add_trace(go.Scatter(
        x=np.r_[x, x[0]], y=np.r_[y, y[0]],
        mode="lines",
        line=dict(color="lightgray", width=1),
        showlegend=False,
    ))

for metric, angle in zip(METRICS, angles):
    cos_a, sin_a = np.cos(angle), np.sin(angle)

    fig.add_trace(go.Scatter(
        x=[0, 1.0 * cos_a], y=[0, 1.0 * sin_a],
        mode="lines",
        line=dict(color="lightgray", width=1),
        showlegend=False,
    ))

    # leader line out to the label
    fig.add_trace(go.Scatter(
        x=[LEADER_INNER * cos_a, LEADER_OUTER * cos_a],
        y=[LEADER_INNER * sin_a, LEADER_OUTER * sin_a],
        mode="lines",
        line=dict(color="#888", width=1),
        showlegend=False,
    ))

    lo, hi = plot_ranges[metric]
    mid = (lo + hi) / 2

    fig.add_annotation(
        x=LABEL_R * cos_a, y=LABEL_R * sin_a,
        text=f"<b>{METRIC_LABELS.get(metric, metric)}</b>",
        showarrow=False,
        font=dict(size=11, color="#1f2d3d"),
        bgcolor="rgba(255,255,255,0.85)",
        align="center",
    )

    inner_r = 0.10

    TICK_STYLES = [
    (inner_r, lo,  "blue", "white", 8, False),
    (0.5,     mid, "#555555", "white", 7, False),
    (1.0,     hi,  "white", "darkblue", 10, True),
    ]

    for r, label_value, color, bg, sze, bold in TICK_STYLES:
        text = f"<b>{label_value:,.2f}</b>" if bold else f"{label_value:,.2f}"

        fig.add_annotation(
            x=r * cos_a, y=r * sin_a,
            text = f"<b>{label_value:,.2f}<b>",
            showarrow=False,
            font=dict(size=sze, color=color),
            bgcolor = bg,
            xshift=20 * cos_a,
            yshift=12 * sin_a,
            align="center",
            opacity = 0.85,
            )

plot_rows = scaled.dropna(subset=METRICS)
stage_colors = pc.sample_colorscale(
    STAGE_COLORMAP, np.linspace(0, 1, len(plot_rows))
)

for (i, row), color in zip(plot_rows.iterrows(), stage_colors):
    r = row[METRICS].to_numpy(dtype=float)
    x, y = xy(r)

    fillcolor = color.replace("rgb(", "rgba(").replace(")", f", {FILL_OPACITY})")

    fig.add_trace(go.Scatter(
        x=np.r_[x, x[0]], y=np.r_[y, y[0]],
        mode="lines+markers",
        name=row["stage"],
        line=dict(color=color, width=2),
        marker=dict(color=color, size=6),
        fill="tonext",
        fillcolor=fillcolor,
    ))

fig.update_layout(
    width=900, height=900,
    xaxis=dict(visible=False, range=[-1.5, 1.5]),
    yaxis=dict(visible=False, range=[-1.65, 1.65], scaleanchor="x", scaleratio=0.9),
    plot_bgcolor = "white",
    showlegend = True,
    font = dict(family="Atkinson Hyperlegible, sans-serif", size=12),
)
fig.show()